In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
clusters = pd.read_csv(f'{indir}/csv/label_transfer/snmct_xgboost_v7.csv', sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters

In [ ]:
ips_subtypes_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/snm3c_ips_subtype.csv'
ips_subtypes = pd.read_csv(ips_subtypes_path, sep='\t', index_col=0)
ips_subtypes.shape

In [ ]:
ips_subtypes.head()

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/processed/rna_finegrain.h5ad')

In [ ]:
adata

In [ ]:
adata.obs['ips_subtype'] = ips_subtypes.reindex(adata.obs.index)['ips_subtype']
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['pred_coarse']

In [ ]:
sc.pl.umap(adata, color=['time', 'line'], ncols=2)

In [ ]:
sc.pl.umap(adata, color=['cluster', 'ips_subtype'], ncols=2)

In [ ]:
sc.pl.umap(adata, color=['ips_subtype'], groups='IPS_alt')

In [ ]:
sc.pl.umap(adata, color=['ips_subtype'], groups='IPS_main')

### write

In [ ]:
umap_df = pd.DataFrame(index=adata.obs.index)
umap_df['umap0'] = adata.obsm['X_umap'][:,0]
umap_df['umap1'] = adata.obsm['X_umap'][:,1]
umap_df['cluster'] = clusters.reindex(umap_df.index)['pred_coarse']
umap_df['donor'] = clusters.reindex(umap_df.index)['donor']
umap_df['ips_subtype'] = ips_subtypes.reindex(umap_df.index)['ips_subtype']

In [ ]:
umap_df.head()

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5'
umap_df.to_csv(f'{indir}/snmct_rna_ips_subtype_umap.csv', sep='\t')

In [ ]:
sc.pl.umap(adata, color=['ips'], ncols=2)

In [ ]:
!date